In [ ]:
import sys
sys.path.append("../qfb_optimization/")
sys.path.append("..")
# Add REGCOIL executable to path before importing replicate_lgradb
sys.path.append("~/regcoil")

import latexplot
latexplot.set_cmap(4)
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import simsopt
import joblib
from joblib import Memory
from scipy.spatial.distance import cdist
from scipy.stats import linregress
from pathlib import Path
import pandas as pd
from simsopt.configs import get_QUASR_data
from simsopt.mhd import vmec_diagnostics
from vmecpp.simsopt_compat import Vmec
from replicate_lgradb.find_single_l import find_regcoil_distance


SINGLE_STAGE_PATH = Path.home() / "single-stage-opt/"


def coil_surf_distance(curves, lcfs) -> np.ndarray:
    pointcloud1 = lcfs.gamma().reshape((-1, 3))
    distances = [np.min(cdist(pointcloud1, c.gamma()), axis=0) for c in curves]
    return np.array(distances).T


def compute_coil_surf_dist(simsopt_filename):
    surfaces, coils = simsopt.load(simsopt_filename)
    lcfs = surfaces[-1].to_RZFourier()

    curves = [c.curve for c in coils]
    return coil_surf_distance(curves, lcfs)

def vs_plot(x_data, y_data, labels=None):
    x_vals, x_label = x_data
    y_vals, y_label = y_data
    title = x_label + " vs " + y_label

    if len(np.shape(y_vals)) >= 2:
        y_vals = np.array(y_vals).T
    elif len(np.shape(y_vals)) == 1:
        y_vals = np.reshape(y_vals, (1,) + np.shape(y_vals))

    assert len(x_vals) == len(y_vals[0])
    if labels is not None:
        assert len(labels) == len(y_vals), f"{len(labels)} != {np.shape(y_vals)}"

    # Linear fit
    # TODO this Fails because some values are inf!!
    for i, y in enumerate(y_vals):
        plt.scatter(x_vals, y, label=title if labels is None else labels[i], s=4)
        reg = linregress(x_vals, y)
        plt.axline(
            xy1=(0, reg.intercept),
            slope=reg.slope,
            color="k" if len(y_vals) == 1 else plt.rcParams["axes.prop_cycle"].by_key()["color"][i],
            label=f"Linear fit {i}: $R^2$ = {reg.rvalue**2:.3f}",
        )

    plt.xlabel(x_label)
    plt.ylabel(y_label)
    plt.title(title)
    if np.min(y_vals) >= 0:
        plt.gca().set_ylim(bottom=0)
    plt.gca().set_xlim(left=0)
    plt.grid(True)
    plt.legend()


In [ ]:
df:pd.DataFrame = pd.read_pickle("../quasr_exploration/QUASR_db/QUASR_08072024.pkl")

# Select the subset for analysis. In the thesis we present results using two different filters. 
# A dedicated copy of the script with exact parameters used to generate the clustering results is in the plots/cluster_by_coils/ folder.

# Filter df by constant number of coils
df = df[df["nc_per_hp"] * df["nfp"] == 6]
df = df.sample(n=6000, replace=False, random_state=1324)

# To get the nfp clustering results, select 
# df["$n_{coils}$"] = 2*(df["nc_per_hp"] * df["nfp"]).astype(int)
# df = pd.concat([
#     df[df["$n_{coils}$"]==4 ].sample(n=200, replace=False, random_state=42),
#     df[df["$n_{coils}$"]==6 ].sample(n=200, replace=False, random_state=42),
#     df[df["$n_{coils}$"]==24].sample(n=200, replace=False, random_state=42),
# ])

In [ ]:
ids = df["ID"].tolist()
parallel = joblib.parallel.Parallel(backend="threading", return_as="generator")
parallel_download_gen = parallel(joblib.parallel.delayed(get_QUASR_data)(idx) for idx in ids)
valid_results_gen = filter(lambda nested: nested[1][1] is not None, zip(ids, parallel_download_gen))
# Flatten the nested tuple 
results = [(nested[0], *nested[1]) for nested in valid_results_gen]

In [ ]:
def normalize_scale(surface, constant_minor=True):
  # Scaling factor for either constant minor or major radius
  if constant_minor:
    scaling = 1.704 / surface.minor_radius()
  else: 
    scaling = 1.0 / surface.major_radius()
  return scaling

In [ ]:
location = './.cachedir'
memory = Memory(location, verbose=0)


def lgradbsc(computed):
    gradB = np.array([[computed.grad_B__XX, computed.grad_B__YX, computed.grad_B__ZX],
                      [computed.grad_B__XY, computed.grad_B__YY, computed.grad_B__ZY],
                      [computed.grad_B__XZ, computed.grad_B__YZ, computed.grad_B__ZZ]])
    scalarGradB = np.sqrt(gradB[0,0] **2 + gradB[1,1]**2 + gradB[2,2] **2)
    LgradBs = (
        np.sqrt(2)
        * computed.modB
        / np.linalg.norm(gradB, ord="fro", axis=(0, 1))
    )
    LgradBscalar = (
        computed.modB / scalarGradB
    )
    # "$L_{\\max \\sigma}$"
    LgradB2maxsigma = (
        computed.modB
        / np.linalg.norm(gradB, ord=2, axis=(0, 1))
    )
    # "$L_{\\sigma}$"
    # TODO: Not entirely sure if the authors meant the nuclear norm (sum of singular values) or this norm (root mean square of singular values)
    LgradB2sigma = (
        np.sqrt(2)
        * computed.modB.flatten()
        # / np.linalg.norm(gradB, ord="nuc", axis=(0, 1))
        / np.sqrt(np.sum(np.linalg.svd(gradB.reshape((3,3,-1)).T, compute_uv=False)**2, axis=1))
    )
    Lgradbgradb = (
        computed.modB * computed.modB / np.linalg.norm(np.sum(np.array([computed.B_X, computed.B_Y, computed.B_Z]) * gradB, axis=0), axis=0)
    )

    def fsa(sqrtg,thing):
        return np.sum(sqrtg * thing)/np.sum(sqrtg)
    sqrtg = np.ascontiguousarray((computed.sqrt_g_vmec).squeeze())
    Bmod = np.ascontiguousarray((computed.modB).squeeze())
    fsa_B = fsa(sqrtg.T.squeeze(), Bmod.T.squeeze()) # Flux surface averaged B
    fsa_B_metric = np.ascontiguousarray((fsa_B * np.sqrt(2)) / (computed.norm_grad_B.squeeze().T* computed.L_reference))[:,:,None]
    #  ["$L^*_{\\nabla \\vec{B}}$", 
    #   "$L_{\\vec{B} \\cdot \\nabla \\vec{B}}$", 
    #   "$L_{\\nabla |B|}$", 
    #   "$L_{\\max \\sigma}$",
    #   "$L_{\\sigma}$",
    #   "$L_{fsa_B}$"]
    return  np.array([ np.min(LgradBs), np.min(Lgradbgradb), np.min(LgradBscalar), np.min(LgradB2maxsigma), np.min(LgradB2sigma),
                       np.min(fsa_B_metric)
                      ])

@memory.cache(ignore=["vmec"])
def vmec_lgradbsc(vmec:Vmec, cache_invalidator:int|str):
    # try:
    #     vmec.run(max_threads=4)
    # except Exception as e:
    #     print(f"Error! {str(e)}")
    #     return np.zeros(6)
    s = [1]
    ntheta = 128
    nphi = 128
    theta = np.linspace(0, 2 * np.pi, ntheta)
    phi = np.linspace(0, 2 * np.pi / vmec.boundary.nfp, nphi)
    # data = vmec_diagnostics.vmec_compute_geometry(vmec_diagnostics.vmec_splines(vmec), s, theta, phi)
    # B0 = np.mean(data.modB)
    # vmec.indata.phiedge = 5.865 * vmec.indata.phiedge / B0
    vmec.recompute_bell()
    try:
        vmec.run(max_threads=4)
        computed = vmec_diagnostics.vmec_compute_geometry(vmec_diagnostics.vmec_splines(vmec), s, theta, phi)
        return lgradbsc(computed)
    except Exception as e:
        print(f"VMEC didn't converge on the second attempt! {str(e)}")
    return np.zeros(6)
    

In [ ]:
from simsopt import geo
import joblib
lgradb_variants = {}
regcoil_distances = {}
num=0

def one_config(idx, surfs, coils):
    global num
    num += 1
    s : geo.SurfaceRZFourier = surfs[-1].to_RZFourier().copy()

    # If the configurations are all scaled to the same major radius instead of minor radius, the results that follow are not qualitatively different
    scaling = normalize_scale(s, constant_minor=True)
    s.rc *= scaling
    s.zs *= scaling
    s.recompute_bell()
    vmec = Vmec("input.preset", keep_all_files=True, verbose=True)
    assert vmec.indata is not None
    vmec.indata.nfp = vmec.boundary.nfp
    vmec.boundary = s
    lgradb_variants[idx] = vmec_lgradbsc(vmec, idx)
    print(num, "... R0:", s.major_radius(), s.minor_radius(), "LgradB:", lgradb_variants[idx])
    assert len(lgradb_variants[idx]) == 6
    if np.allclose(lgradb_variants[idx], 0): 
        regcoil_distances[ f"{idx:07d}" ] = 0
    else:
        # Must pass 
        vmec.run()
        regcoil_distances[ f"{idx:07d}" ] = find_regcoil_distance(vmec, idx)
    print("Lregcoil", regcoil_distances[ f"{idx:07d}" ])


for idx, surfs, coils in results:
    one_config(idx, surfs, coils)

In [ ]:
a = set([f"{idx:07d}" for idx in lgradb_variants.keys()])
b = set(regcoil_distances.keys())
a == b

In [ ]:
def vs_plot2(x_data, y_data, labels=None):
    x_vals, x_label = x_data
    y_vals, y_label = y_data
    title = x_label + " vs " + y_label
    filename = title.replace(" ", "_") + ".csv"

    # Ensure y_vals has shape (n_series, n_points)
    if len(np.shape(y_vals)) >= 2:
        y_vals = np.array(y_vals).T
    elif len(np.shape(y_vals)) == 1:
        y_vals = np.reshape(y_vals, (1,) + np.shape(y_vals))

    assert len(x_vals) == len(y_vals[0])
    if labels is not None:
        assert len(labels) == len(y_vals), f"{len(labels)} != {np.shape(y_vals)}"

    # Build DataFrame
    plot_df = pd.DataFrame({x_label: x_vals})
    for i, y in enumerate(y_vals):
        col_name = labels[i] if labels is not None else f"{y_label}_{i}"
        plot_df[col_name] = y

    # Save to CSV
    plot_df.to_csv(filename, index=False)
    return plot_df  # return in case you want to use it in memory too

In [ ]:
cmap10 = False
regcoil_plot = True
bdistrib_plot = False
latexplot.set_cmap(4)
if cmap10: 
    plt.rcParams['axes.prop_cycle'] = matplotlib.cycler(color=[plt.get_cmap('tab10')(e) for e in range(4)])

lgradB_names = ["$L^*_{\\nabla \\vec{B}}$", 
                "$L_{\\vec{B} \\cdot \\nabla \\vec{B}}$", 
                "$L_{\\nabla |B|}$", 
                "$L_{\\max \\sigma}$",
                "$L_{\\sigma}$",
                "$L_{fsab}$"
                ]
LgradB_keyed = {}
coil_surf_dist = {}

for idx, surfs, coils in results:
    filename =  f"{idx:07d}" 
    print("Loading", filename)
    if idx in lgradb_variants:
        computed = lgradb_variants[idx]
    else:
        # VMEC didn't converge for this configuration, so its not a valid result
        continue

    # $L_{REGCOIL}$
    simsopt_name = filename.replace("input.", "serial").replace(
        "_output.h5", ".json"
    )
    LgradB_keyed[simsopt_name] = lgradb_variants[idx]
    LgradB = LgradB_keyed[simsopt_name][0]

    # The distances here were verified with the QUASR database GUI and are correct.
    # simsopt_path = f"{SINGLE_STAGE_PATH}/quasr_exploration/QUASR_db/simsopt_serials/{simsopt_name[6:10]}/{simsopt_name}"
    coil_surf_dist[simsopt_name] = coil_surf_distance([c.curve for c in coils], surfs[-1]) * normalize_scale(surfs[-1], constant_minor=True) #compute_coil_surf_dist(simsopt_path)
    
    print(
        simsopt_name,
        "has minimum filament coil distance",
        np.min(coil_surf_dist[simsopt_name]),
    )

#########################

# Extract filenames and corresponding values for plotting
latexplot.set_cmap(len(lgradB_names))
if cmap10: 
    plt.rcParams['axes.prop_cycle'] = matplotlib.cycler(color=[plt.get_cmap('tab10')(e) for e in range(4)])
validlgradb_dict = {key: value for key, value in LgradB_keyed.items() if np.all(value > 0.0)} #
filenames = list(validlgradb_dict.keys())

bothLgradBandRegcoil = list(validlgradb_dict.keys())
if regcoil_plot:
    validr_dict = {key: value for key, value in regcoil_distances.items() if (value != 0) and np.isfinite(value)}
    bothLgradBandRegcoil = list(set(validlgradb_dict.keys()).intersection(validr_dict.keys()))
    regcoil_vals = (np.array([regcoil_distances[f] for f in bothLgradBandRegcoil]), "$L_{REGCOIL}$")
    
    LgradB_vals = (np.array([LgradB_keyed[f][:len(lgradB_names)] for f in bothLgradBandRegcoil]), "$L^*_{\\nabla B}$") 
    coil_min_vals = (
        [np.min(coil_surf_dist[f]) for f in bothLgradBandRegcoil],
        "QUASR coil distance",
    )
    # latexplot.figure()
    vs_plot2(regcoil_vals, LgradB_vals, lgradB_names)
    # latexplot.savenshow("regcoil_vs_lgradb")
    # latexplot.figure()
    vs_plot2(coil_min_vals, regcoil_vals)
    # latexplot.savenshow("regcoil_vs_quasr")



coil_min_vals = (
    np.array([np.min(coil_surf_dist[f]) for f in bothLgradBandRegcoil]),
    "QUASR coil distance",
)

# Do we want to plot the same points as for regcoil?
LgradB_vals = (np.array([LgradB_keyed[f] for f in filenames]), "$L^*_{\\nabla B}$") 
coil_min_vals = (
    np.array([np.min(coil_surf_dist[f]) for f in filenames]),
    "QUASR coil distance",
)

if cmap10: 
    plt.rcParams['axes.prop_cycle'] = matplotlib.cycler(color=[plt.get_cmap('tab10')(e) for e in range(len(lgradB_names))])

#########################
# latexplot.figure()
vs_plot2(coil_min_vals, LgradB_vals, lgradB_names)
# latexplot.savenshow("lgradb_vs_quasr")

In [ ]:
dfexport = pd.DataFrame(
  data=lgradb_variants.keys(),
  columns=["QUASR ID"]
)
for i, name in enumerate(lgradB_names):
  dfexport[name] = LgradB_vals[0][:,i]

dfexport["coil_min_distance"] = coil_min_vals[0]


std_deviations = []
for idx, surfs, coils in results
  arr = np.array([coil.current.get_value() for coil in coils])
  std_deviations.append(np.std(arr)/np.mean(np.abs(arr)))

dfexport["coil current std deviation"] = std_deviations
dfexport.to_csv("lgradb_on_quasr_kappel.csv", index=False)

In [ ]:
import pickle

with open("lgradb_vs_quasr2.pkl", "wb") as f:
    pickle.dump((coil_min_vals, LgradB_vals, lgradB_names), f)

vs_plot(coil_min_vals, LgradB_vals, lgradB_names)
plt.savefig("lgradb_vs_quasr2.png", dpi=300, bbox_inches="tight")

In [ ]:
# Get the data for some of the outliers from LgradB
df["$L_{\\nabla |B|}$"] = df["$L^*_{\\nabla B}$"].map(lambda x: 0 if np.any(np.isnan(x)) else x[2])
df[(df["$L_{\\nabla |B|}$"]<5) & (df["QUASR coil distance"]>4) & (df["QUASR coil distance"]<16)]

# Failiure cases

In [ ]:
import pickle

with open("lgradb_vs_quasr2.pkl", "rb") as f:
    (coil_min_vals, LgradB_vals, lgradB_names) = pickle.load(f)
vs_plot2(coil_min_vals, LgradB_vals, lgradB_names)